# 02 · Scenario walkthrough

Run the full stress model across all seven scenarios, inspect the mixed financing stack, compute breakpoints, and visualize the contagion / private-credit channels.

> Run with the project venv kernel: `uv run jupyter lab`.
> This is a **stress-testing tool, not a forecast** — many inputs are low-confidence assumptions.

In [ ]:
import sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
import pandas as pd
import matplotlib.pyplot as plt

from src.model.scenarios import load_all_scenarios, load_scenario
from src.model.datacenter_project import build_representative_project
from src.model.private_credit_module import build_private_credit_book
from src.model.contagion_module import build_contagion_model
from src.model.treasury_module import build_treasury_module
from src.model.stress_metrics import run_scenario, compute_breakpoints, sensitivity_table, refinancing_gap

In [ ]:
project = build_representative_project(capacity_mw=100.0)
pc = build_private_credit_book()
cg = build_contagion_model()
tm = build_treasury_module()
scenarios = load_all_scenarios()

## 1. The mixed financing stack

Explicitly NOT 'all floating-rate private credit'.

In [ ]:
base = load_scenario('base_case')
rows = [{
    'source': s.name, 'amount': s.amount, 'type': s.fixed_or_floating.value,
    'ref': s.base_rate_reference, 'spread_bps': s.credit_spread_bps,
    'all_in_rate': round(s.all_in_rate(base), 4), 'maturity': s.maturity_year,
    'hedge': s.hedge_ratio, 'swap': s.swapped_to_fixed_ratio, 'lender': s.lender_type.value,
} for s in project.financing_stack.sources]
stack_df = pd.DataFrame(rows)
stack_df

In [ ]:
fs = project.financing_stack
print(f"WACD: {fs.weighted_average_cost_of_debt(base):.2%}")
print(f"Floating share GROSS: {fs.floating_rate_share(net_of_hedges=False):.1%}")
print(f"Floating share NET (after hedges/swaps): {fs.floating_rate_share(net_of_hedges=True):.1%}")

## 2. Coverage ratios across all scenarios

In [ ]:
rows = []
for name, sc in scenarios.items():
    m = project.metrics(sc)
    rows.append({'scenario': name, 'DSCR': m['dscr'], 'ICR': m['icr'],
                 'EBITDA_margin': m['ebitda_margin'], 'FCF_after_capex': m['fcf_after_capex'],
                 'WACD': fs.weighted_average_cost_of_debt(sc)})
metrics_df = pd.DataFrame(rows).set_index('scenario')
metrics_df.round(3)

In [ ]:
ax = metrics_df[['DSCR', 'ICR']].plot(kind='bar', figsize=(11, 4))
thr = __import__('src.config', fromlist=['assumption_value']).assumption_value('thresholds', {})
ax.axhline(thr.get('dscr_min', 1.2), color='red', ls='--', label='DSCR min')
ax.set_title('DSCR / ICR by scenario'); ax.legend(); plt.tight_layout(); plt.show()

## 3. Breakpoint panel

One-axis sensitivity from the base case: how far each lever can move before a danger threshold is crossed (holding all else at base).

In [ ]:
bp = compute_breakpoints(project)
for k, v in bp.items():
    if k not in ('thresholds_used', 'base_levels'):
        print(f'{k:48s}: {v}')

## 4. Sensitivity sweep (SOFR)

In [ ]:
sens = pd.DataFrame(sensitivity_table(project, axis='sofr_bps'))
sens.round(3)

## 5. Private credit: MTM vs realized loss + investor allocation

These are **distinct concepts** and must not be summed. All private-credit figures are LOW-CONFIDENCE.

In [ ]:
sev = load_scenario('severe_private_credit_liquidity_crunch')
print(f"Expected (realized) loss: ${pc.expected_loss(sev)/1e9:,.1f}B")
print(f"Mark-to-market loss:      ${pc.mark_to_market_loss(sev)/1e9:,.1f}B")
print(f"Liquidity stress score:   {pc.liquidity_stress_score(sev):.2f}")
alloc = pc.investor_loss_allocation(sev)
pd.Series({k: v/1e9 for k, v in alloc.items()}, name='loss_$B').sort_values(ascending=False).to_frame()

## 6. Contagion: stylized transmission heatmap

A stylized linear exposure-matrix cascade — **not a forecast**.

In [ ]:
shock = cg.build_initial_shock_from_scenario(sev, pc_book=pc, project=project)
heat = cg.systemic_risk_heatmap(shock)
print(f"Funding contraction estimate: ${cg.funding_contraction_estimate(shock)/1e9:,.1f}B")
print(f"Capex slowdown estimate:      {cg.capex_slowdown_estimate(shock):.1%}")
heat.sort_values('total_loss', ascending=False).head(12)

In [ ]:
h = heat.sort_values('total_loss', ascending=True)
ax = h['total_loss'].plot(kind='barh', figsize=(9, 6), color='crimson')
ax.set_title('Contagion: total propagated loss by node (stylized)'); ax.set_xlabel('USD'); plt.tight_layout(); plt.show()

## 7. Refinancing wall (combined-severe)

In [ ]:
gap = refinancing_gap(project, scenario=load_scenario('combined_severe_case'))
pd.DataFrame(gap).T

---
**Reminder:** this is a stress-testing tool, not a forecasting oracle. Private-credit data is opaque, SPV/off-balance-sheet exposures are incomplete, AI-specific capex is often not separately disclosed, and many values are scenario assumptions rather than facts.